# NOVA Unified Intelligence Research
Research-only notebook: XGBoost + genuine PyTorch LSTM + SMC + immutable NOVA Brain replay + OOF meta-decision. CPU-first for Colab/Kaggle.

**No training runs by default.** Do not enable training until a provenance-verified dataset, session-aware coverage receipt and broker-cost fixture are approved. This notebook cannot submit orders.

Final OOS starts at `2026-07-01T00:00:00Z`; development tuning must remain strictly before that boundary.

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, random, warnings
import numpy as np
import pandas as pd
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'workers').is_dir():
    candidates = [Path('/content/VOOM_SYSTEM-NOVA-AI'), Path('/kaggle/working/VOOM_SYSTEM-NOVA-AI')]
    REPO_ROOT = next((p for p in candidates if (p / 'workers').is_dir()), REPO_ROOT)
if (REPO_ROOT / 'workers').is_dir(): sys.path.insert(0, str(REPO_ROOT))
SEED = 42
random.seed(SEED); np.random.seed(SEED)
DATA_PATH = Path('/content/drive/MyDrive/NOVA/approved-market-data/XAUUSDm_H1.csv')
MANIFEST_PATH = DATA_PATH.with_suffix('.manifest.json')
OUTPUT_DIR = Path('/content/drive/MyDrive/NOVA/research-runs/unified-intelligence')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TRAINING = False  # explicit opt-in only after all gates are reviewed
RUN_LSTM = False
FINAL_OOS_START = pd.Timestamp('2026-07-01T00:00:00Z')
HORIZON = 5
N_SPLITS = 5
print({'training_enabled': RUN_TRAINING, 'repo_root': str(REPO_ROOT), 'data_exists': DATA_PATH.is_file(), 'manifest_exists': MANIFEST_PATH.is_file()})

## 1A. Authenticated Datacenter export — optional, fail closedThis cell downloads one exact, approved Datacenter dataset version through the existing NOVA Remote MCP Gateway. It does not use a public artifact URL or service-role key. Store a short-lived Supabase user access token in Colab Secrets as `NOVA_SUPABASE_ACCESS_TOKEN`; never paste it into notebook source or output. Set `NOVA_DATASET_ID` and `NOVA_DATASET_VERSION` as environment variables, then explicitly set `IMPORT_APPROVED_DATASET=True`.This requires Datacenter PR #6 and NOVA Gateway PR #423 to be reviewed, merged, and deployed, with the fixed `DATACENTER_MCP_URL` configured on the Gateway. Until then, export must remain blocked.

In [ ]:
import requestsfrom datetime import timezone# The NOVA Gateway URL is fixed to the existing authenticated endpoint.NOVA_GATEWAY_MCP_URL = 'https://voom-system-mcp.suwapat787.workers.dev/mcp'IMPORT_APPROVED_DATASET = False  # explicit opt-in; does not enable model trainingNOVA_DATASET_ID = os.getenv('NOVA_DATASET_ID', '').strip()NOVA_DATASET_VERSION = int(os.getenv('NOVA_DATASET_VERSION', '0') or 0)EXPORT_CHUNK_SIZE = 1000  # Gateway and Datacenter both cap this at 1,000 rowsdef _get_user_access_token():    token = os.getenv('NOVA_SUPABASE_ACCESS_TOKEN', '').strip()    if not token:        try:            from google.colab import userdata            token = (userdata.get('NOVA_SUPABASE_ACCESS_TOKEN') or '').strip()        except Exception:            token = ''    if not token:        raise RuntimeError('BLOCKED: add a short-lived Supabase user token to Colab Secrets as NOVA_SUPABASE_ACCESS_TOKEN.')    if not token.lower().startswith('ey'):        raise RuntimeError('BLOCKED: NOVA_SUPABASE_ACCESS_TOKEN does not look like a user JWT.')    return tokendef _gateway_rpc(payload, token, protocol=False):    headers = {        'Authorization': f'Bearer {token}',        'Content-Type': 'application/json',        'Accept': 'application/json, text/event-stream',    }    if protocol:        headers['MCP-Protocol-Version'] = '2025-03-26'    response = requests.post(NOVA_GATEWAY_MCP_URL, headers=headers, json=payload, timeout=60)    response.raise_for_status()    return response.json() if response.content else {}def _call_nova_dataset_export(token, dataset_id, version, offset, limit):    initialized = _gateway_rpc({        'jsonrpc': '2.0', 'id': 'nova-notebook-init', 'method': 'initialize',        'params': {'protocolVersion': '2025-03-26', 'capabilities': {},                   'clientInfo': {'name': 'nova-unified-research-notebook', 'version': '1.0'}},    }, token)    if initialized.get('error'):        raise RuntimeError('NOVA Gateway MCP initialization failed')    notice = requests.post(        NOVA_GATEWAY_MCP_URL,        headers={'Authorization': f'Bearer {token}', 'Content-Type': 'application/json',                 'Accept': 'application/json, text/event-stream',                 'MCP-Protocol-Version': '2025-03-26'},        json={'jsonrpc': '2.0', 'method': 'notifications/initialized', 'params': {}},        timeout=60,    )    if notice.status_code not in (200, 202):        notice.raise_for_status()    response = _gateway_rpc({        'jsonrpc': '2.0', 'id': 'nova-dataset-export', 'method': 'tools/call',        'params': {'name': 'dataset_export_chunk',                   'arguments': {'dataset_id': dataset_id, 'version': version,                                 'offset': offset, 'limit': limit}},    }, token, protocol=True)    payload = response.get('result') or {}    if response.get('error') or payload.get('isError') is True:        details = payload.get('content', [{}])[0].get('text', 'Gateway export failed')        raise RuntimeError(f'Datacenter export blocked: {details}')    blocks = payload.get('content', [])    text_block = next((item.get('text') for item in blocks                       if item.get('type') == 'text' and isinstance(item.get('text'), str)), None)    if not text_block:        raise RuntimeError('Datacenter export returned no JSON payload')    return json.loads(text_block)def _canonical_export_chunk(rows):    normalized = []    for source_row in rows:        row = dict(source_row)        stamp = pd.Timestamp(row.get('timestamp'))        if stamp.tzinfo is None:            raise ValueError('Exported timestamps must be timezone-aware')        row['timestamp'] = stamp.tz_convert('UTC').to_pydatetime().isoformat()        normalized.append(row)    return json.dumps(normalized, sort_keys=True, separators=(',', ':'), ensure_ascii=False)def import_approved_datacenter_dataset():    if not NOVA_DATASET_ID or NOVA_DATASET_VERSION < 1:        raise RuntimeError('BLOCKED: set NOVA_DATASET_ID and NOVA_DATASET_VERSION to an exact immutable version.')    token = _get_user_access_token()    offset = 0    rows = []    chunk_hashes = []    first = None    while True:        chunk = _call_nova_dataset_export(            token, NOVA_DATASET_ID, NOVA_DATASET_VERSION, offset, EXPORT_CHUNK_SIZE        )        if chunk.get('status') != 'ok':            raise RuntimeError('Datacenter did not return status=ok')        if chunk.get('dataset_id') != NOVA_DATASET_ID or chunk.get('version') != NOVA_DATASET_VERSION:            raise RuntimeError('Dataset identity/version changed during export')        if chunk.get('offset') != offset or chunk.get('returned_rows') != len(chunk.get('rows', [])):            raise RuntimeError('Dataset export offset or row-count continuity failure')        if first is None:            first = chunk            if chunk.get('dataset_status') != 'PASSED' or chunk.get('approval_status') != 'approved':                raise RuntimeError('BLOCKED: dataset is not PASSED and explicitly approved')            if chunk.get('quality_receipt', {}).get('status') != 'PASSED':                raise RuntimeError('BLOCKED: persisted quality receipt is not PASSED')            source_manifest = chunk.get('manifest') or {}            if (source_manifest.get('symbols') != ['XAUUSDm']                    or source_manifest.get('timeframe') != 'H1'                    or source_manifest.get('timezone') != 'UTC'                    or source_manifest.get('expected_interval_seconds') != 3600                    or source_manifest.get('source') != 'exness-mt5-demo-readonly'):                raise RuntimeError('BLOCKED: source manifest is not the approved Exness XAUUSDm H1 UTC contract')        else:            for key in ('artifact_checksum', 'row_count', 'dataset_id', 'version'):                if chunk.get(key) != first.get(key):                    raise RuntimeError(f'Dataset export changed pinned field: {key}')        actual_chunk_hash = hashlib.sha256(            _canonical_export_chunk(chunk.get('rows', [])).encode('utf-8')        ).hexdigest()        if actual_chunk_hash != chunk.get('chunk_sha256'):            raise RuntimeError('Dataset chunk SHA-256 mismatch')        if not chunk.get('rows'):            raise RuntimeError('Dataset export returned an empty non-terminal chunk')        rows.extend(chunk['rows'])        chunk_hashes.append(chunk['chunk_sha256'])        next_offset = chunk.get('next_offset')        expected_next = offset + len(chunk['rows'])        if next_offset is None:            if chunk.get('complete') is not True:                raise RuntimeError('Dataset export ended without a completion receipt')            break        if next_offset != expected_next or next_offset <= offset:            raise RuntimeError('Dataset export next_offset is not contiguous')        offset = next_offset    row_count = int(first['row_count'])    if len(rows) != row_count:        raise RuntimeError(f'Dataset export incomplete: expected {row_count}, received {len(rows)}')    source_manifest = first['manifest']    quality_report = first['quality_receipt'].get('report') or {}    missing = int(quality_report.get('missing_interval_count', 0))    irregular = int(quality_report.get('irregular_interval_count', 0))    if missing or irregular:        raise RuntimeError('BLOCKED: Datacenter quality receipt still reports unresolved gaps/irregular intervals')    frame = pd.DataFrame(rows)    required = {'timestamp', 'open', 'high', 'low', 'close', 'tick_volume', 'spread'}    if not required.issubset(frame.columns):        raise RuntimeError(f'Exported data missing required columns: {sorted(required - set(frame.columns))}')    frame['timestamp'] = pd.to_datetime(frame['timestamp'], utc=True, errors='raise')    if frame.timestamp.duplicated().any() or not frame.timestamp.is_monotonic_increasing:        raise RuntimeError('Exported timestamps are duplicated or out of order')    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)    temp_data = DATA_PATH.with_suffix(DATA_PATH.suffix + '.part')    temp_manifest = MANIFEST_PATH.with_suffix(MANIFEST_PATH.suffix + '.part')    frame.to_csv(temp_data, index=False, lineterminator='\n')    materialized_sha = hashlib.sha256(temp_data.read_bytes()).hexdigest()    interval = int(source_manifest['expected_interval_seconds'])    actual_end = pd.Timestamp(source_manifest['coverage_end']) + pd.Timedelta(seconds=interval)    normalized_manifest = {        'dataset_id': NOVA_DATASET_ID,        'dataset_version': NOVA_DATASET_VERSION,        'provider': source_manifest['source'],        'symbol': source_manifest['symbols'][0],        'timeframe': source_manifest['timeframe'],        'timezone': 'UTC',        'actual_start_utc': pd.Timestamp(source_manifest['coverage_start']).isoformat().replace('+00:00', 'Z'),        'actual_end_utc': actual_end.isoformat().replace('+00:00', 'Z'),        'row_count': row_count,        'completeness_status': 'PASS',        'training_eligible': True,        'ordersEnabled': False,        'executionEnabled': False,        'liveTradingEnabled': False,        'content_sha256': materialized_sha,        'sha256': materialized_sha,        'source_artifact_sha256': first['artifact_checksum'],        'source_artifact_format': first.get('artifact_format'),        'quality_receipt_id': first['quality_receipt']['receipt_id'],        'gap_validation': {'status': 'PASS', 'missing_interval_count': missing,                           'irregular_interval_count': irregular},        'export_receipt': {'chunk_size': EXPORT_CHUNK_SIZE, 'chunk_count': len(chunk_hashes),                           'chunk_sha256': chunk_hashes},        'exported_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),    }    temp_manifest.write_text(json.dumps(normalized_manifest, indent=2), encoding='utf-8')    os.replace(temp_data, DATA_PATH)    os.replace(temp_manifest, MANIFEST_PATH)    return {'dataset_id': NOVA_DATASET_ID, 'version': NOVA_DATASET_VERSION,            'rows': row_count, 'source_artifact_sha256': first['artifact_checksum'],            'materialized_csv_sha256': materialized_sha, 'chunks': len(chunk_hashes),            'data_path': str(DATA_PATH), 'manifest_path': str(MANIFEST_PATH)}if IMPORT_APPROVED_DATASET:    IMPORT_RECEIPT = import_approved_datacenter_dataset()    print(json.dumps(IMPORT_RECEIPT, indent=2))else:    print('Dataset import disabled by default; set IMPORT_APPROVED_DATASET=True after PRs are deployed.')

## 2. Dataset admission — fail closedThe manifest must pin the exact Datacenter dataset ID/version, source, symbol/timeframe, UTC half-open coverage, row count, exported CSV SHA-256, source Parquet SHA-256, persisted quality receipt and read-only flags. Missing costs are not treated as zero.

In [ ]:
from workers.quant_engine.unified_evaluation import (
    validate_dataset_manifest,
    expanding_purged_walk_forward,
    validate_point_in_time_rows,
    require_immutable_llm_replay,
    fuse_research_signals,
    split_development_final_oos,
    assign_session_segments,
)

def load_approved_dataset(data_path=DATA_PATH, manifest_path=MANIFEST_PATH):
    if not data_path.is_file() or not manifest_path.is_file():
        raise RuntimeError('BLOCKED: approved dataset or manifest is absent; do not train on synthetic data.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    digest = validate_dataset_manifest(manifest, data_path)
    frame = pd.read_csv(data_path)
    required = ['timestamp', 'open', 'high', 'low', 'close', 'tick_volume', 'spread']
    missing = [c for c in required if c not in frame.columns]
    if missing:
        raise ValueError(f'Missing source columns: {missing}')
    frame['timestamp'] = pd.to_datetime(frame['timestamp'], utc=True, errors='raise')
    validate_point_in_time_rows(frame['timestamp'].tolist())
    for col in ['open', 'high', 'low', 'close', 'tick_volume', 'spread']:
        frame[col] = pd.to_numeric(frame[col], errors='raise')
    numeric = frame[['open', 'high', 'low', 'close', 'tick_volume', 'spread']].to_numpy()
    if not np.isfinite(numeric).all():
        raise ValueError('Non-finite market values')
    if (frame.tick_volume < 0).any() or (frame.spread < 0).any():
        raise ValueError('Negative volume/spread')
    invalid = (
        (frame.high < frame.low)
        | (frame.open < frame.low) | (frame.open > frame.high)
        | (frame.close < frame.low) | (frame.close > frame.high)
    )
    if invalid.any():
        raise ValueError('Invalid OHLC bounds')
    if manifest.get('symbol') != 'XAUUSDm' or manifest.get('timeframe') != 'H1':
        raise ValueError('This research notebook admits only XAUUSDm H1')
    interval = manifest.get('expected_interval_seconds')
    if interval != 3600:
        raise ValueError('XAUUSDm H1 manifest must declare a 3600-second cadence')
    frame['segment_id'] = assign_session_segments(frame['timestamp'].tolist(), interval)
    if len(frame) != int(manifest['row_count']):
        raise ValueError('Manifest row_count mismatch')
    if frame.timestamp.min() < pd.Timestamp(manifest['actual_start_utc']) or frame.timestamp.max() >= pd.Timestamp(manifest['actual_end_utc']):
        raise ValueError('Observed rows fall outside manifest UTC range')
    if manifest.get('gap_validation', {}).get('status') != 'PASS':
        raise ValueError('BLOCKED: session-aware gap receipt is not PASS')
    return frame, manifest, digest

if DATA_PATH.is_file() and MANIFEST_PATH.is_file():
    market, dataset_manifest, DATA_SHA256 = load_approved_dataset()
    print('Dataset admitted:', len(market), DATA_SHA256)
else:
    market = None
    print('BLOCKED: no approved dataset artifact; training and performance evaluation are not run.')


## 2. Causal features and forward labels
Features use only current/past closed bars. The decision is formed at bar close and the simulator enters no earlier than the next eligible bar. The last `HORIZON` rows have unresolved labels and are dropped, never coerced to class zero.

In [ ]:
def _features_one_segment(frame):
    x = pd.DataFrame(index=frame.index)
    close = frame.close.astype(float)
    ret = close.pct_change()
    x['ret_1'] = ret
    x['ret_5'] = close.pct_change(5)
    x['volatility_20'] = ret.rolling(20, min_periods=20).std()
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).rolling(14, min_periods=14).mean()
    x['rsi_14'] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    ema12 = close.ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = close.ewm(span=26, adjust=False, min_periods=26).mean()
    x['macd'] = ema12 - ema26
    x['volume_ratio_20'] = frame.tick_volume / frame.tick_volume.rolling(20, min_periods=20).mean().replace(0, np.nan)
    x['spread_points'] = frame.spread.astype(float)
    return x

def make_causal_features(frame):
    parts = [_features_one_segment(group) for _, group in frame.groupby('segment_id', sort=False)]
    return pd.concat(parts).sort_index().replace([np.inf, -np.inf], np.nan)

def make_forward_labels(frame, horizon=HORIZON):
    future = frame.groupby('segment_id', sort=False).close.shift(-horizon)
    label = (future > frame.close).astype('float')
    label[future.isna()] = np.nan
    return label

if market is not None:
    features = make_causal_features(market)
    labels = make_forward_labels(market)
    valid = features.notna().all(axis=1) & labels.notna()
    research = market.loc[valid].copy()
    research['label_end_timestamp'] = market.groupby('segment_id', sort=False).timestamp.shift(-HORIZON).loc[valid]
    X = features.loc[valid].copy()
    y = labels.loc[valid].astype(int)
    development_mask, oos_mask, boundary_purged = split_development_final_oos(
        research.timestamp.tolist(), research.label_end_timestamp.tolist(), FINAL_OOS_START
    )
    development_mask = pd.Series(development_mask, index=research.index)
    oos_mask = pd.Series(oos_mask, index=research.index)
    print('Rows purged because forward labels cross final OOS boundary:', sum(boundary_purged))
    dev_X, dev_y = X.loc[development_mask], y.loc[development_mask]
    oos_X, oos_y = X.loc[oos_mask], y.loc[oos_mask]
    if dev_X.empty or oos_X.empty: raise RuntimeError('Development or sealed OOS partition is empty')
    print('Development rows:', len(dev_X), '| sealed OOS rows:', len(oos_X))

## 3. Development-only walk-forward and base models
Every fold trains only on earlier rows; the forward-label horizon is purged before validation. Feature scaling, calibration and meta-learning are fit only inside development folds.

In [ ]:
def get_folds(n_rows):
    return expanding_purged_walk_forward(n_rows, n_splits=N_SPLITS, min_train_size=max(500, n_rows//3), horizon=HORIZON)

def train_xgboost_fold(x_train, y_train, x_valid, y_valid):
    from xgboost import XGBClassifier
    model = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, eval_metric='logloss', random_state=SEED, n_jobs=2)
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)
    return model, model.predict_proba(x_valid)[:,1]

def train_lstm_fold(x_train, y_train, x_valid, y_valid, train_segments, valid_segments, epochs=20, patience=4):
    import torch
    from torch import nn
    torch.manual_seed(SEED)
    class CausalLSTM(nn.Module):
        def __init__(self, n_features, hidden=32):
            super().__init__()
            self.lstm = nn.LSTM(n_features, hidden, batch_first=True)
            self.dropout = nn.Dropout(0.1)
            self.head = nn.Linear(hidden, 1)
        def forward(self, z):
            _, (h, _) = self.lstm(z)
            return self.head(self.dropout(h[-1])).squeeze(-1)
    def windows(a, segments, seq=32):
        a = np.asarray(a, dtype=np.float32)
        segments = np.asarray(segments)
        ends = [i for i in range(seq-1, len(a)) if np.all(segments[i-seq+1:i+1] == segments[i])]
        if not ends:
            return np.empty((0,seq,a.shape[1]),dtype=np.float32), np.asarray([],dtype=int)
        return np.stack([a[i-seq+1:i+1] for i in ends]), np.asarray(ends,dtype=int)
    tx, train_ends = windows(x_train, train_segments)
    vx, valid_ends = windows(x_valid, valid_segments)
    ty = np.asarray(y_train, dtype=np.float32)[train_ends]
    vy = np.asarray(y_valid, dtype=np.float32)[valid_ends]
    if len(tx)==0 or len(vx)==0:
        raise ValueError('Insufficient contiguous sequence rows')
    model = CausalLSTM(tx.shape[-1])
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    best, best_state, wait, history = float('inf'), None, 0, []
    for epoch in range(epochs):
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(torch.from_numpy(tx)), torch.from_numpy(ty))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        model.eval()
        with torch.no_grad():
            val = float(loss_fn(model(torch.from_numpy(vx)), torch.from_numpy(vy)).item())
        history.append({'epoch':epoch+1,'validation_loss':val})
        if val < best:
            best = val
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
        if wait >= patience:
            break
    if best_state is None:
        raise RuntimeError('LSTM did not produce a valid checkpoint')
    model.load_state_dict(best_state)
    with torch.no_grad():
        probs = torch.sigmoid(model(torch.from_numpy(vx))).numpy()
    import io
    buf = io.BytesIO()
    torch.save(best_state, buf)
    checkpoint = buf.getvalue()
    return model, probs, history, hashlib.sha256(checkpoint).hexdigest(), valid_ends

print('Walk-forward folds are generated only after dataset admission; no fitting occurs in this cell.')


## 4. SMC and NOVA Brain replay contracts
SMC must be the canonical confirmed-candle implementation; the 4H bias is available only after the 4H candle closes and pivots only after right-side confirmation. NOVA Brain contributes only immutable historical responses that existed by the decision time. No LLM call is made during historical replay.

In [ ]:
def validate_smc_event(event, decision_time):
    required = ['event_time','available_at','signal','source_version','confirmed']
    if any(k not in event for k in required): raise ValueError('SMC event provenance incomplete')
    if pd.Timestamp(event['event_time']) > pd.Timestamp(decision_time): raise ValueError('SMC event time is after decision time')
    if not event['confirmed'] or pd.Timestamp(event['available_at']) > pd.Timestamp(decision_time):
        raise ValueError('SMC pivot/HTF event was not yet available at decision time')
    if event['signal'] not in (-1,0,1): raise ValueError('Invalid SMC signal')
    return event

def validate_brain_replay(record, decision_time):
    return require_immutable_llm_replay(record, pd.Timestamp(decision_time))

# Load SMC and NOVA Brain ledgers only from separately approved, immutable artifacts.
SMC_LEDGER_PATH = OUTPUT_DIR / 'smc_events.jsonl'
BRAIN_LEDGER_PATH = OUTPUT_DIR / 'brain_replay.jsonl'
print('No SMC/LLM replay is inferred when the immutable ledgers are absent.')

## 5. Meta Decision Layer — OOF only
Train a meta-model on out-of-fold development predictions only. Never train it on in-sample base-model predictions. The final OOS is opened only after base models, calibrators, thresholds and meta-model are frozen. Missing, stale or contradictory evidence means ABSTAIN.

In [ ]:
from workers.quant_engine.unified_evaluation import fuse_research_signals
def build_meta_features(xgb_p, lstm_p, smc_signal, brain_signal):
    return np.column_stack([xgb_p, lstm_p, smc_signal, brain_signal])

def fit_meta_model(oof_features, oof_labels):
    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=SEED)
    model.fit(oof_features, oof_labels)
    return model

def meta_decision(meta_model, xgb_p, lstm_p, smc_signal, brain_signal):
    row = build_meta_features([xgb_p],[lstm_p],[smc_signal],[brain_signal])
    p = float(meta_model.predict_proba(row)[0,1])
    return fuse_research_signals(xgb_p,lstm_p,smc_signal,brain_signal,p,min_meta_probability=0.60)

print('Meta-model fit is prohibited until OOF development predictions have been generated and audited.')

## 6. Shared cost-aware evaluator
All candidates must use the same next-eligible-bar fill, symbol specification and cost assumptions. The evaluator must reject missing spread/commission/swap/slippage fixtures rather than treating unknown costs as zero. Bar-only data cannot claim tick-level fill accuracy.

In [ ]:
def net_trade_pnl(gross_pnl, spread_cost, commission, swap, slippage):
    costs = [gross_pnl, spread_cost, commission, swap, slippage]
    if not np.isfinite(costs).all(): raise ValueError('Non-finite PnL/cost input')
    if min(spread_cost, commission, slippage) < 0: raise ValueError('Costs cannot be negative')
    return float(gross_pnl - spread_cost - commission - swap - slippage)

def summarize_trade_ledger(trades):
    if not trades: return {'closed_trades':0,'net_pnl':0.0,'profit_factor':None,'max_drawdown':None}
    net = np.asarray([float(t['net_pnl']) for t in trades], dtype=float)
    if not np.isfinite(net).all(): raise ValueError('Invalid trade ledger')
    gains, losses = net[net>0].sum(), -net[net<0].sum()
    equity = np.cumsum(net); peak = np.maximum.accumulate(np.r_[0.0,equity])[1:]
    dd = peak - equity
    return {'closed_trades':int(len(net)),'net_pnl':float(net.sum()),'profit_factor':float(gains/losses) if losses>0 else None,'max_drawdown':float(dd.max()) if len(dd) else 0.0,'expectancy_per_trade':float(net.mean())}

print('Cost model must be populated from versioned Exness XAUUSDm Demo specifications before any strategy replay.')

## 7. Explicit training gate and run receipt
The notebook will not fit a model, evaluate OOS, or write a model artifact while data admission is blocked. Set `RUN_TRAINING=True` only after reviewing the exact dataset manifest and costs. This is not permission to promote or trade.

In [ ]:
def read_jsonl(path):
    if not path.is_file(): raise RuntimeError(f'BLOCKED: required immutable ledger missing: {path}')
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def signal_maps(smc_records, brain_records):
    smc_map, brain_map = {}, {}
    for event in smc_records:
        raw_key = event.get('decision_timestamp', event.get('available_at'))
        if raw_key is None: raise ValueError('SMC replay record has no decision/availability timestamp')
        key = pd.Timestamp(raw_key)
        validate_smc_event(event, key)
        if key in smc_map: raise ValueError('Duplicate SMC decision timestamp')
        smc_map[key] = int(event['signal'])
    for record in brain_records:
        raw_key = record.get('decision_timestamp', record.get('timestamp'))
        if raw_key is None: raise ValueError('NOVA Brain replay record has no decision timestamp')
        key = pd.Timestamp(raw_key)
        validate_brain_replay(record, key)
        decision = str(record['decision']).upper()
        brain_map[key] = 1 if decision in ('LONG','BUY','BULLISH') else -1 if decision in ('SHORT','SELL','BEARISH') else 0 if decision in ('HOLD','NEUTRAL','ABSTAIN') else None
        if brain_map[key] is None: raise ValueError(f'Unknown NOVA Brain decision: {decision}')
    return smc_map, brain_map

def aligned_signals(timestamps, smc_map, brain_map):
    keys = [pd.Timestamp(t) for t in timestamps]
    missing_smc = [str(t) for t in keys if t not in smc_map]
    missing_brain = [str(t) for t in keys if t not in brain_map]
    if missing_smc or missing_brain:
        raise RuntimeError(f'BLOCKED: replay coverage missing; SMC={len(missing_smc)}, Brain={len(missing_brain)}')
    return np.asarray([smc_map[t] for t in keys]), np.asarray([brain_map[t] for t in keys])

def run_training_and_oos():
    if not RUN_TRAINING or not RUN_LSTM: raise RuntimeError('Set RUN_TRAINING=True and RUN_LSTM=True only after approval.')
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, brier_score_loss
    from xgboost import XGBClassifier
    import torch
    smc_map, brain_map = signal_maps(read_jsonl(SMC_LEDGER_PATH), read_jsonl(BRAIN_LEDGER_PATH))
    dev_frame = research.loc[development_mask].reset_index(drop=True)
    oos_frame = research.loc[oos_mask].reset_index(drop=True)
    dx = dev_X.reset_index(drop=True); dy = dev_y.reset_index(drop=True)
    ox = oos_X.reset_index(drop=True); oy = oos_y.reset_index(drop=True)
    oof_rows, oof_labels = [], []
    for fold_no, fold in enumerate(get_folds(len(dx)), start=1):
        tr = slice(fold.train_start, fold.train_end); va = slice(fold.valid_start, fold.valid_end)
        xtr, ytr = dx.iloc[tr], dy.iloc[tr]
        xva, yva = dx.iloc[va], dy.iloc[va]
        xgb_model, xgb_p = train_xgboost_fold(xtr, ytr, xva, yva)
        scaler = StandardScaler().fit(xtr)
        lstm_model, lstm_p, history, ckpt_sha, valid_ends = train_lstm_fold(scaler.transform(xtr), ytr, scaler.transform(xva), yva, dev_frame.segment_id.iloc[fold.train_start:fold.train_end], dev_frame.segment_id.iloc[fold.valid_start:fold.valid_end])
        positions = np.arange(fold.valid_start, fold.valid_end)[valid_ends]
        if len(positions) != len(lstm_p): raise RuntimeError('LSTM OOF alignment failure')
        smc_s, brain_s = aligned_signals(dev_frame.timestamp.iloc[positions], smc_map, brain_map)
        xgb_aligned = xgb_p[valid_ends]
        oof_rows.extend(np.column_stack([xgb_aligned, lstm_p, smc_s, brain_s]).tolist())
        oof_labels.extend(dy.iloc[positions].astype(int).tolist())
        print(f'Fold {fold_no}: train={len(xtr)} valid={len(xva)} OOF={len(positions)} LSTM_best_val_loss={min(h["validation_loss"] for h in history):.6f} checkpoint_sha256={ckpt_sha}')
    if len(oof_rows) < 100: raise RuntimeError('Insufficient OOF meta-training rows')
    meta_model = LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=SEED)
    meta_model.fit(np.asarray(oof_rows), np.asarray(oof_labels))
    # Final base models: all XGBoost development rows; LSTM early stopping uses only a development tail.
    final_xgb = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0, eval_metric='logloss', random_state=SEED, n_jobs=2)
    final_xgb.fit(dx, dy)
    split = int(len(dx) * 0.8); fit_end = split - HORIZON
    if fit_end <= 32 or len(dx) - split <= 32: raise RuntimeError('Insufficient development rows for final LSTM fit/validation')
    final_scaler = StandardScaler().fit(dx.iloc[:fit_end])
    final_lstm, _, final_history, final_ckpt_sha, final_valid_ends = train_lstm_fold(final_scaler.transform(dx.iloc[:fit_end]), dy.iloc[:fit_end], final_scaler.transform(dx.iloc[split:]), dy.iloc[split:], dev_frame.segment_id.iloc[:fit_end], dev_frame.segment_id.iloc[split:])
    xgb_oos_p = final_xgb.predict_proba(ox)[:,1]
    # Carry the last 31 known development rows as context; no OOS label is used for fitting.
    context = final_scaler.transform(dx.iloc[-31:])
    oos_scaled = final_scaler.transform(ox)
    combined = np.vstack([context, oos_scaled]).astype(np.float32)
    combined_segments = np.r_[dev_frame.segment_id.iloc[-31:].to_numpy(), oos_frame.segment_id.to_numpy()]
    ends = [i for i in range(31, len(combined)) if np.all(combined_segments[i-31:i+1] == combined_segments[i])]
    if not ends: raise RuntimeError('No contiguous LSTM OOS sequences after session segmentation')
    lstm_windows = np.stack([combined[i-31:i+1] for i in ends])
    oos_positions = np.asarray(ends, dtype=int) - 31
    if len(final_valid_ends) == 0: raise RuntimeError('Final LSTM validation produced no contiguous sequences')
    final_lstm.eval()
    with torch.no_grad(): lstm_oos_p = torch.sigmoid(final_lstm(torch.from_numpy(lstm_windows))).numpy()
    xgb_oos_p = xgb_oos_p[oos_positions]
    oos_frame = oos_frame.iloc[oos_positions].reset_index(drop=True)
    oy = oy.iloc[oos_positions].reset_index(drop=True)
    smc_oos, brain_oos = aligned_signals(oos_frame.timestamp, smc_map, brain_map)
    meta_x = np.column_stack([xgb_oos_p, lstm_oos_p, smc_oos, brain_oos])
    meta_p = meta_model.predict_proba(meta_x)[:,1]
    decisions = [fuse_research_signals(float(a),float(b),int(c),int(d),float(p),0.60)['decision'] for a,b,c,d,p in zip(xgb_oos_p,lstm_oos_p,smc_oos,brain_oos,meta_p)]
    mask = np.asarray([d != 'ABSTAIN' for d in decisions])
    pred = np.asarray([1 if d == 'LONG' else 0 for d in decisions])[mask]
    truth = oy.to_numpy(dtype=int)[mask]
    metrics = {'oos_rows':int(len(oy)),'decision_coverage':float(mask.mean()),'selective_rows':int(mask.sum()),'selective_accuracy':float(accuracy_score(truth,pred)) if len(truth) else None,'selective_balanced_accuracy':float(balanced_accuracy_score(truth,pred)) if len(truth) else None,'selective_f1':float(f1_score(truth,pred,zero_division=0)) if len(truth) else None,'meta_brier':float(brier_score_loss(oy.to_numpy(dtype=int),meta_p)),'net_pnl':'NOT_MEASURED','profit_factor':'NOT_MEASURED','max_drawdown':'NOT_MEASURED'}
    import joblib
    joblib.dump(final_xgb, OUTPUT_DIR/'xgboost_candidate.joblib')
    joblib.dump(final_scaler, OUTPUT_DIR/'lstm_scaler.joblib')
    joblib.dump(meta_model, OUTPUT_DIR/'meta_decision_candidate.joblib')
    torch.save(final_lstm.state_dict(), OUTPUT_DIR/'lstm_candidate.pt')
    artifact_hashes = {p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in OUTPUT_DIR.iterdir() if p.is_file() and p.name.endswith(('.joblib','.pt'))}
    receipt = {'status':'CLASSIFICATION_OOS_COMPLETED_NOT_PROFITABILITY_EVIDENCE','dataset_sha256':DATA_SHA256,'final_oos_start_utc':FINAL_OOS_START.isoformat(),'metrics':metrics,'artifact_sha256':artifact_hashes,'ordersEnabled':False,'executionEnabled':False,'liveTradingEnabled':False}
    (OUTPUT_DIR/'run_receipt.json').write_text(json.dumps(receipt,indent=2),encoding='utf-8')
    return receipt

if not RUN_TRAINING:
    print('BLOCKED BY DEFAULT: no fitting or artifact writes have occurred.')
elif market is None:
    raise RuntimeError('Training refused: approved dataset was not admitted.')
else:
    RUN_RESULT = run_training_and_oos()
    print(json.dumps(RUN_RESULT,indent=2))
